In [1]:
ENV["PYTHONPATH"] = "/home/gridsan/aligho/.local/lib/python3.8/site-packages/PyNormaliz-2.15-py3.8-linux-x86_64.egg";

In [212]:
using DelimitedFiles, PyPlot, LinearAlgebra
using Crystalline, Brillouin, MPBUtils, SymmetryBases
using Crystalline: TEST_αβγs, TEST_αβγ, dot, norm
topology_paper_dir = "../TopologyPaper/"
include(topology_paper_dir * "get-freqs-symeigs.jl")
include(topology_paper_dir * "symeigs-from-io.jl");

### First, we find all space groups with multidimensional irreps (with and without TR), saving them in vectors `filtered_sgnums` and `filtered_sgnums_tr`

In [360]:
filtered_sgnums = Int64[] # Space groups with multidimensional irreps at one or more k-points (even without TR)
filtered_sgnums_dict = Dict{Int64, Vector{String}}() # A dictionary storing a space group number as a key and the k-labels corresponding to multidimensional irreps as the value

filtered_sgnums_tr = Int64[] # Same as the above but with time-reversal symmetry 
filtered_sgnums_dict_tr = Dict{Int64, Vector{String}}() # Same as the above but with time-reversal symmetry 

for sgnum in 1:230 # 230 space groups in 3D
    sgnum_lgirreps = lgirreps(sgnum) # Irreps at each high-symmetry k-point
    good_klabs = String[] # List of k-labels corresponding to k-points with multidimensional irreps
    for (k_lab, irreps) in sgnum_lgirreps
        good_irrep = false
        for irrep in irreps
            irrep_dim = first(size(first(irrep.matrices)))
            irrep_dim > 1 || continue # Check that irrep dimension is larger than one 
            push!(filtered_sgnums, sgnum)
            good_irrep = true
        end
        good_irrep && push!(good_klabs, k_lab)
    end
    !isempty(good_klabs) && push!(filtered_sgnums_dict, sgnum => good_klabs)
end
unique!(filtered_sgnums); 

# Run the same loop as above but with time reversal symmetry 
for sgnum in 1:230 
    sgnum_lgirreps = lgirreps(sgnum)
    good_klabs = String[] 
    for (k_lab, irreps) in sgnum_lgirreps
        good_irrep = false
        for irrep in realify(irreps)
            irrep_dim = first(size(first(irrep.matrices)))
            irrep_dim > 1 || continue # Check that irrep dimension is larger than one 
            push!(filtered_sgnums_tr, sgnum)
            good_irrep = true
        end
        good_irrep && push!(good_klabs, k_lab)
    end
    !isempty(good_klabs) && push!(filtered_sgnums_dict_tr, sgnum => good_klabs)
end
unique!(filtered_sgnums);
unique!(filtered_sgnums_tr);

print("Number of spacegroups with multidimensional irreps (even without time reversal symmetry): ", length(filtered_sgnums), "\n")
print("Number of spacegroups with multidimensional irreps (with time reversal symmetry): ", length(filtered_sgnums_tr), "\n")

Number of spacegroups with multidimensional irreps (even without time reversal symmetry): 175
Number of spacegroups with multidimensional irreps (with time reversal symmetry): 209


In [361]:
sgnum_pairs_primitive = Dict{Int, Vector{Int}}()
sgnum_pairs_primitive_tr = Dict{Int, Vector{Int}}()

for sgnum_1 in filtered_sgnums 
    sgnum_1_group = primitivize(spacegroup(sgnum_1)) # put in primitive setting
    filtered_group_operations = filter(x->iszero(x.translation), sgnum_1_group) # All group operations which are fully symmorphic 
    pg_1 = find_isomorphic_parent_pointgroup(filtered_group_operations)[1] # Find point group isomorphic to the group corresponding to all symmorphic operations 
    sgnum1_vec = Int[] # Vector of candidate subgroups of sgnum_1
    sgnum_1_c_system = crystalsystem(primitivize(directbasis(sgnum_1), centering(sgnum_1))) # Whether the system is cubic, tetragonal, hexagonal, etc 
    for sgnum_2 in 1:230
        sgnum_2_c_system = crystalsystem(primitivize(directbasis(sgnum_2), centering(sgnum_2)))
        (sgnum_2_c_system == sgnum_1_c_system) || continue # Require that the crystal system remain the same
        sgnum_2_group = primitivize(spacegroup(sgnum_2))
        (filter(x->iszero(x.translation), sgnum_2_group) == sgnum_2_group) || continue # Only look at symmorphic space groups
        pg_2 = find_isomorphic_parent_pointgroup(sgnum_2_group)[1]
        (pg_1 == pg_2) || continue # Make sure that parent point group of sgnum_2 is the same as the parent point group of sgnum_1 with only symmorphic operations included
        push!(sgnum1_vec, sgnum_2)
    end
    !(isempty(sgnum1_vec)) || continue
    any(x -> x in filtered_sgnums, sgnum1_vec) || continue
    push!(sgnum_pairs_primitive, sgnum_1 => sgnum1_vec)
end

# Same loop as above but with time reversal symmetry

for sgnum_1 in filtered_sgnums_tr 
    sgnum_1_group = primitivize(spacegroup(sgnum_1)) # put in primitive setting
    filtered_group_operations = filter(x->iszero(x.translation), sgnum_1_group) # All group operations which are fully symmorphic 
    pg_1 = find_isomorphic_parent_pointgroup(filtered_group_operations)[1] # Find point group isomorphic to the group corresponding to all symmorphic operations 
    sgnum1_vec = Int[] # Vector of candidate subgroups of sgnum_1
    sgnum_1_c_system = crystalsystem(primitivize(directbasis(sgnum_1), centering(sgnum_1))) # Whether the system is cubic, tetragonal, hexagonal, etc 
    for sgnum_2 in 1:230
        sgnum_2_c_system = crystalsystem(primitivize(directbasis(sgnum_2), centering(sgnum_2)))
        (sgnum_2_c_system == sgnum_1_c_system) || continue # Require that the crystal system remain the same
        sgnum_2_group = primitivize(spacegroup(sgnum_2))
        (filter(x->iszero(x.translation), sgnum_2_group) == sgnum_2_group) || continue # Only look at symmorphic space groups
        pg_2 = find_isomorphic_parent_pointgroup(sgnum_2_group)[1]
        (pg_1 == pg_2) || continue 
        push!(sgnum1_vec, sgnum_2)
    end
    !(isempty(sgnum1_vec)) || continue
    any(x -> x in filtered_sgnums_tr, sgnum1_vec) || continue
    push!(sgnum_pairs_primitive_tr, sgnum_1 => sgnum1_vec)
end

In [362]:
#filter(x -> !(x in keys(sgnum_pairs_primitive_tr)), keys(sgnum_pairs_primitive))

In [363]:
#filter(x -> !(x in keys(sgnum_pairs_primitive)), keys(sgnum_pairs_primitive_tr))

In [364]:
# If the original spacegroup is in the value of the (key, value) pair, we obviously just replace the value with a one element vector [key]
for (key, value) in sgnum_pairs_primitive
    if key in value
        sgnum_pairs_primitive[key] = [key]
    end
end

more_than_one_dict = filter(x -> length(x[2])>1, sgnum_pairs_primitive)

for (sgnum_1, sgnum_2v) in more_than_one_dict
    sg1_primitivized = primitivize(spacegroup(sgnum_1))
    sgnum_1_symmorphic_elements = filter(x->iszero(x.translation), sg1_primitivized)
    for sgnum_2 in sgnum_2v
        sg2_primitivized = primitivize(spacegroup(sgnum_2))
        (filter(x -> x in sg2_primitivized, sgnum_1_symmorphic_elements) == sgnum_1_symmorphic_elements) || continue
        sgnum_pairs_primitive[sgnum_1] = [sgnum_2]
    end
end

In [365]:
# If the original spacegroup is in the value of the (key, value) pair, we obviously just replace the value with a one element vector [key]
for (key, value) in sgnum_pairs_primitive_tr
    if key in value
        sgnum_pairs_primitive_tr[key] = [key]
    end
end
more_than_one_dict = filter(x -> length(x[2])>1, sgnum_pairs_primitive_tr)
for (sgnum_1, sgnum_2v) in more_than_one_dict
    sg1_primitivized = primitivize(spacegroup(sgnum_1))
    sgnum_1_symmorphic_elements = filter(x->iszero(x.translation), sg1_primitivized)
    for sgnum_2 in sgnum_2v
        sg2_primitivized = primitivize(spacegroup(sgnum_2))
        (filter(x -> x in sg2_primitivized, sgnum_1_symmorphic_elements) == sgnum_1_symmorphic_elements) || continue
        sgnum_pairs_primitive_tr[sgnum_1] = [sgnum_2]
    end
end

In [366]:
@assert isempty(filter(x->!(x in sgnum_pairs_primitive_tr), sgnum_pairs_primitive))
@assert isempty(filter(x->length(x[2])>1, sgnum_pairs_primitive_tr))
@assert isempty(filter(x->length(x[2])>1, sgnum_pairs_primitive))

### As a sanity check, we look at space groups that change upon putting the defect and make sure the lattice vectors of the parent and child spacegroups make sense

In [367]:
for (key, val) in filter(x->!(x[2][1] == x[1]), sgnum_pairs_primitive)
    basis_1 = primitivize(directbasis(key), centering(key))
    basis_2 = primitivize(directbasis(val[1]), centering(val[1]))
    println("Parent: $(key)", " Child: $(val[1])")
    println(round.(hcat(basis_1...), digits=3))
    println(round.(hcat(basis_2...), digits=3), "\n")
end

Parent: 223 Child: 200
[1.0 0.0 0.0; 0.0 1.0 0.0; 0.0 0.0 1.0]
[1.0 0.0 0.0; 0.0 1.0 0.0; 0.0 0.0 1.0]

Parent: 188 Child: 149
[1.0 -0.5 0.0; 0.0 0.866 0.0; 0.0 0.0 1.607]
[1.0 -0.5 0.0; 0.0 0.866 0.0; 0.0 0.0 1.308]

Parent: 190 Child: 150
[1.0 -0.5 0.0; 0.0 0.866 0.0; 0.0 0.0 0.8]
[1.0 -0.5 0.0; 0.0 0.866 0.0; 0.0 0.0 0.523]

Parent: 219 Child: 196
[0.0 0.5 0.5; 0.5 0.0 0.5; 0.5 0.5 0.0]
[0.0 0.5 0.5; 0.5 0.0 0.5; 0.5 0.5 0.0]

Parent: 182 Child: 150
[1.0 -0.5 0.0; 0.0 0.866 0.0; 0.0 0.0 1.067]
[1.0 -0.5 0.0; 0.0 0.866 0.0; 0.0 0.0 1.513]

Parent: 227 Child: 166
[0.0 0.5 0.5; 0.5 0.0 0.5; 0.5 0.5 0.0]
[0.5 -0.5 -0.0; 0.289 0.289 -0.577; 0.191 0.191 0.191]

Parent: 186 Child: 156
[1.0 -0.5 0.0; 0.0 0.866 0.0; 0.0 0.0 1.986]
[1.0 -0.5 0.0; 0.0 0.866 0.0; 0.0 0.0 0.767]

Parent: 185 Child: 157
[1.0 -0.5 0.0; 0.0 0.866 0.0; 0.0 0.0 1.822]
[1.0 -0.5 0.0; 0.0 0.866 0.0; 0.0 0.0 1.407]

Parent: 210 Child: 196
[0.0 0.5 0.5; 0.5 0.0 0.5; 0.5 0.5 0.0]
[0.0 0.5 0.5; 0.5 0.0 0.5; 0.5 0.5 0.0]

P

### The only one that seems questionable in the above is spacegroup 227. In the github, we have notes that show this is fine.

In [374]:
sorted_keys = sort(collect(keys(sgnum_pairs_primitive)));

In [376]:
### Find the crystal system for each
for key in sorted_keys
    key_system = crystalsystem(primitivize(directbasis(key), centering(key))) # Whether the system is cubic, tetragonal, hexagonal, etc 
    println("Sgnum: $key, reduced space group: $(sgnum_pairs_primitive[key]) system: ", key_system)
end

Sgnum: 89, reduced space group: [89] system: tetragonal
Sgnum: 97, reduced space group: [97] system: triclinic
Sgnum: 99, reduced space group: [99] system: tetragonal
Sgnum: 107, reduced space group: [107] system: triclinic
Sgnum: 111, reduced space group: [111] system: tetragonal
Sgnum: 115, reduced space group: [115] system: tetragonal
Sgnum: 119, reduced space group: [119] system: triclinic
Sgnum: 121, reduced space group: [121] system: triclinic
Sgnum: 123, reduced space group: [123] system: tetragonal
Sgnum: 139, reduced space group: [139] system: triclinic
Sgnum: 149, reduced space group: [149] system: hexagonal
Sgnum: 150, reduced space group: [150] system: hexagonal
Sgnum: 155, reduced space group: [155] system: trigonal
Sgnum: 156, reduced space group: [156] system: hexagonal
Sgnum: 157, reduced space group: [157] system: hexagonal
Sgnum: 160, reduced space group: [160] system: trigonal
Sgnum: 162, reduced space group: [162] system: hexagonal
Sgnum: 164, reduced space group: [

### Below, we find the maximum and minimum irrep dimensions for the spacegroup we reduce to after addition of the defect. We do this both for spacegroups filtered in the presence of time reversal and those without time reversal

In [384]:
for key in sorted_keys
    sgnum_1 = key
    sgnum_2 = sgnum_pairs_primitive[key][1]
    irreps_at_gamma = lgirreps(sgnum_2)["Γ"]
    irreps_at_gamma_tr = realify(irreps_at_gamma)
    irrep_dims = [first(size(first(irrep_at_gamma.matrices))) for irrep_at_gamma in irreps_at_gamma]
    irrep_dims_tr = [first(size(first(irrep_at_gamma_tr.matrices))) for irrep_at_gamma_tr in irreps_at_gamma_tr]
    println("Sgnum: $sgnum_1")
    println("Min dimension (without tr): ", minimum(irrep_dims), "  ", "Max dimension (without tr): ", maximum(irrep_dims))
    println("Min dimension (with tr): ", minimum(irrep_dims_tr), "  ", "Max dimension (with tr): ", maximum(irrep_dims_tr), "\n")
end

Sgnum: 89
Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Sgnum: 97
Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Sgnum: 99
Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Sgnum: 107
Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Sgnum: 111
Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Sgnum: 115
Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Sgnum: 119
Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Sgnum: 121
Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dim

In [222]:
for (key, val) in  sgnum_pairs_primitive_tr
    sgnum_1 = key
    sgnum_2 = val[1]
    irreps_at_gamma = lgirreps(sgnum_2)["Γ"]
    irreps_at_gamma_tr = realify(irreps_at_gamma)
    irrep_dims = [first(size(first(irrep_at_gamma.matrices))) for irrep_at_gamma in irreps_at_gamma]
    irrep_dims_tr = [first(size(first(irrep_at_gamma_tr.matrices))) for irrep_at_gamma_tr in irreps_at_gamma_tr]
    println("Min dimension (without tr): ", minimum(irrep_dims), "  ", "Max dimension (without tr): ", maximum(irrep_dims))
    println("Min dimension (with tr): ", minimum(irrep_dims_tr), "  ", "Max dimension (with tr): ", maximum(irrep_dims_tr), "\n")
end

Min dimension (without tr): 1  Max dimension (without tr): 1
Min dimension (with tr): 1  Max dimension (with tr): 2

Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Min dimension (without tr): 1  Max dimension (without tr): 1
Min dimension (with tr): 1  Max dimension (with tr): 2

Min dimension (without tr): 1  Max dimension (without tr): 1
Min dimension (with tr): 1  Max dimension (with tr): 2

Min dimension (without tr): 1  Max dimension (without tr): 3
Min dimension (with tr): 1  Max dimension (with tr): 3

Min dimension (without tr): 1  Max dimension (without tr): 3
Min dimension (with tr): 1  Max dimension (with tr): 3

Min dimension (without tr): 1  Max dimension (without tr): 3
Min dimension (with tr): 1  Max dimension (with tr): 3

Min dimension (without tr): 1  Max dimension (without tr): 2
Min dimension (with tr): 1  Max dimension (with tr): 2

Min dimension (without tr): 1  Max dimension (without tr): 2
Min

### Now, we look at what happens if we relax the requirement that the parent and child space groups have the same crystal system

In [234]:
sgnum_pairs_primitive_relax = Dict{Int, Vector{Int}}()
sgnum_pairs_primitive_relax_tr = Dict{Int, Vector{Int}}()

for sgnum_1 in filtered_sgnums 
    sgnum_1_group = primitivize(spacegroup(sgnum_1)) # put in primitive setting
    filtered_group_operations = filter(x->iszero(x.translation), sgnum_1_group) # All group operations which are fully symmorphic 
    pg_1 = find_isomorphic_parent_pointgroup(filtered_group_operations)[1] # Find point group isomorphic to the group corresponding to all symmorphic operations 
    sgnum1_vec = Int[] # Vector of candidate subgroups of sgnum_1
    sgnum_1_c_system = crystalsystem(primitivize(directbasis(sgnum_1), centering(sgnum_1))) # Whether the system is cubic, tetragonal, hexagonal, etc 
    for sgnum_2 in 1:230
        sgnum_2_c_system = crystalsystem(primitivize(directbasis(sgnum_2), centering(sgnum_2)))
        #(sgnum_2_c_system == sgnum_1_c_system) || continue # Require that the crystal system remain the same
        sgnum_2_group = primitivize(spacegroup(sgnum_2))
        (filter(x->iszero(x.translation), sgnum_2_group) == sgnum_2_group) || continue # Only look at symmorphic space groups
        pg_2 = find_isomorphic_parent_pointgroup(sgnum_2_group)[1]
        (pg_1 == pg_2) || continue # Make sure that parent point group of sgnum_2 is the same as the parent point group of sgnum_1 with only symmorphic operations included
        push!(sgnum1_vec, sgnum_2)
    end
    !(isempty(sgnum1_vec)) || continue
    any(x -> x in filtered_sgnums, sgnum1_vec) || continue
    push!(sgnum_pairs_primitive_relax, sgnum_1 => sgnum1_vec)
end

# Same loop as above but with time reversal symmetry

for sgnum_1 in filtered_sgnums_tr 
    sgnum_1_group = primitivize(spacegroup(sgnum_1)) # put in primitive setting
    filtered_group_operations = filter(x->iszero(x.translation), sgnum_1_group) # All group operations which are fully symmorphic 
    pg_1 = find_isomorphic_parent_pointgroup(filtered_group_operations)[1] # Find point group isomorphic to the group corresponding to all symmorphic operations 
    sgnum1_vec = Int[] # Vector of candidate subgroups of sgnum_1
    sgnum_1_c_system = crystalsystem(primitivize(directbasis(sgnum_1), centering(sgnum_1))) # Whether the system is cubic, tetragonal, hexagonal, etc 
    for sgnum_2 in 1:230
        sgnum_2_c_system = crystalsystem(primitivize(directbasis(sgnum_2), centering(sgnum_2)))
        #(sgnum_2_c_system == sgnum_1_c_system) || continue # Require that the crystal system remain the same
        sgnum_2_group = primitivize(spacegroup(sgnum_2))
        (filter(x->iszero(x.translation), sgnum_2_group) == sgnum_2_group) || continue # Only look at symmorphic space groups
        pg_2 = find_isomorphic_parent_pointgroup(sgnum_2_group)[1]
        (pg_1 == pg_2) || continue 
        push!(sgnum1_vec, sgnum_2)
    end
    !(isempty(sgnum1_vec)) || continue
    any(x -> x in filtered_sgnums_tr, sgnum1_vec) || continue
    push!(sgnum_pairs_primitive_relax_tr, sgnum_1 => sgnum1_vec)
end

### Below, we print out which space groups are missed if we require the crystal system to be invariant

In [238]:
filter(x-> !(x[1] in keys(sgnum_pairs_primitive)), sgnum_pairs_primitive_relax)

Dict{Int64, Vector{Int64}} with 1 entry:
  224 => [162, 166]

In [239]:
filter(x-> !(x[1] in keys(sgnum_pairs_primitive_tr)), sgnum_pairs_primitive_relax_tr)

Dict{Int64, Vector{Int64}} with 7 entries:
  213 => [143, 146]
  224 => [162, 166]
  201 => [147, 148]
  212 => [143, 146]
  205 => [147, 148]
  198 => [143, 146]
  222 => [147, 148]

### Now, we try something different. For each of the 230 space groups, we find the ones with multidimensional point group irreps 

In [356]:
good_sgnums = Integer[]
good_sgnums_dict = Dict{Integer, String}()
for sgnum in 1:230
    sgnum_group = primitivize(spacegroup(sgnum)) # put in primitive setting
    filtered_group_operations = filter(x->iszero(x.translation), sgnum_group) # All group operations which are fully symmorphic 
    pg = find_isomorphic_parent_pointgroup(filtered_group_operations)[1] # Find point group isomorphic to the group corresponding to all symmorphic operations 
    irreps_parent_point_group = pgirreps(pg.label)
    irrep_sizes = [first(size(first(irrep.matrices))) for irrep in irreps_parent_point_group]
    any(x -> x > 1, irrep_sizes) || continue
    push!(good_sgnums, sgnum)
    push!(good_sgnums_dict, sgnum=>pg.label)
end

In [357]:
@show good_sgnums_dict

good_sgnums_dict = Dict{Integer, String}(123 => "4/mmm", 197 => "23", 215 => "-43m", 219 => "23", 182 => "321", 164 => "-3m1", 115 => "-4m2", 186 => "3m1", 196 => "23", 185 => "31m", 210 => "23", 139 => "4/mmm", 191 => "6/mmm", 207 => "432", 183 => "6mm", 150 => "321", 218 => "23", 224 => "-31m", 177 => "622", 111 => "-42m", 188 => "312", 204 => "m-3", 119 => "-42m", 162 => "-31m", 216 => "-43m", 156 => "3m1", 208 => "23", 194 => "-3m1", 211 => "432", 202 => "m-3", 157 => "31m", 200 => "m-3", 195 => "23", 160 => "3m1", 187 => "-6m2", 217 => "-43m", 189 => "-62m", 227 => "-31m", 107 => "4mm", 225 => "m-3m", 193 => "-31m", 229 => "m-3m", 209 => "432", 226 => "m-3", 223 => "m-3", 221 => "m-3m", 190 => "321", 99 => "4mm", 121 => "-42m", 166 => "-31m", 89 => "422", 149 => "312", 155 => "312", 97 => "422")


Dict{Integer, String} with 54 entries:
  123 => "4/mmm"
  197 => "23"
  215 => "-43m"
  219 => "23"
  182 => "321"
  164 => "-3m1"
  115 => "-4m2"
  186 => "3m1"
  196 => "23"
  185 => "31m"
  210 => "23"
  139 => "4/mmm"
  191 => "6/mmm"
  207 => "432"
  183 => "6mm"
  150 => "321"
  218 => "23"
  224 => "-31m"
  177 => "622"
  111 => "-42m"
  188 => "312"
  204 => "m-3"
  119 => "-42m"
  162 => "-31m"
  216 => "-43m"
  ⋮   => ⋮

### Now we do the same but with time reversal

In [386]:
good_sgnums = Integer[]
good_sgnums_dict = Dict{Integer, String}()
for sgnum in 1:230
    sgnum_group = primitivize(spacegroup(sgnum)) # put in primitive setting
    filtered_group_operations = filter(x->iszero(x.translation), sgnum_group) # All group operations which are fully symmorphic 
    pg = find_isomorphic_parent_pointgroup(filtered_group_operations)[1] # Find point group isomorphic to the group corresponding to all symmorphic operations 
    irreps_parent_point_group = realify(pgirreps(pg.label))
    irrep_sizes = [first(size(first(irrep.matrices))) for irrep in irreps_parent_point_group]
    any(x -> x > 1, irrep_sizes) || continue
    push!(good_sgnums, sgnum)
    push!(good_sgnums_dict, sgnum=>pg.label)
end

In [387]:
good_sgnums_dict

Dict{Integer, String} with 106 entries:
  114 => "-4"
  123 => "4/mmm"
  220 => "3"
  117 => "-4"
  197 => "23"
  215 => "-43m"
  219 => "23"
  182 => "321"
  164 => "-3m1"
  115 => "-4m2"
  186 => "3m1"
  112 => "-4"
  185 => "31m"
  196 => "23"
  210 => "23"
  139 => "4/mmm"
  168 => "6"
  191 => "6/mmm"
  207 => "432"
  104 => "4"
  205 => "-3"
  183 => "6mm"
  158 => "3"
  150 => "321"
  176 => "-3"
  ⋮   => ⋮